# 08 Review Summarization
Generate insights, summaries, and actionable feedback from customer reviews.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Load all processed data
df = pd.read_csv('../data/cleaned_reviews.csv')
print(f'Loaded {len(df)} reviews')

# Basic statistics
total_reviews = len(df)
positive_count = (df['sentiment'] == 1).sum()
negative_count = (df['sentiment'] == 0).sum()
positive_pct = positive_count / total_reviews * 100

print(f'\nTotal Reviews: {total_reviews}')
print(f'Positive: {positive_count} ({positive_pct:.1f}%)')
print(f'Negative: {negative_count} ({100-positive_pct:.1f}%)')

In [ ]:
# Extract key phrases (frequent bigrams)
from collections import Counter
import re

all_bigrams = []
for text in df['reviewText_clean']:
    words = text.split()
    bigrams = [' '.join([words[i], words[i+1]]) for i in range(len(words)-1)]
    all_bigrams.extend(bigrams)

top_bigrams = Counter(all_bigrams).most_common(30)

print('\nTop 30 Phrases:')
for i, (phrase, count) in enumerate(top_bigrams, 1):
    print(f'{i:2d}. {phrase:30s} ({count:4d})')

In [ ]:
# Sentiment by rating level
sentiment_by_rating = df.groupby('overall').agg({
    'sentiment': ['count', 'sum', 'mean']
}).round(3)

sentiment_by_rating.columns = ['Total_Reviews', 'Positive_Count', 'Positive_Rate']
sentiment_by_rating['Positive_Rate'] = (sentiment_by_rating['Positive_Rate'] * 100).round(1)

print('\nSentiment by Rating:')
print(sentiment_by_rating)

In [ ]:
# Find most helpful reviews (by length and rating consistency)
df['text_length'] = df['reviewText_clean'].str.split().str.len()
df['is_long'] = df['text_length'] > df['text_length'].quantile(0.75)

# Positive long reviews
positive_reviews = df[(df['sentiment'] == 1) & (df['is_long'])].nlargest(5, 'text_length')
print('\nTop Positive Reviews:')
for idx, row in positive_reviews.iterrows():
    print(f'\nRating: {row["overall"]}/5')
    print(f'Preview: {row["reviewText_clean"][:150]}...')

In [ ]:
# Generate insights summary
insights = {
    'dataset_summary': {
        'total_reviews': int(total_reviews),
        'positive_reviews': int(positive_count),
        'negative_reviews': int(negative_count),
        'positive_percentage': round(positive_pct, 2),
        'average_rating': float(df['overall'].mean())
    },
    'key_metrics': {
        'avg_review_length': float(df['text_length'].mean()),
        'median_review_length': float(df['text_length'].median()),
        'rating_distribution': df['overall'].value_counts().to_dict()
    },
    'top_phrases': {phrase: count for phrase, count in top_bigrams[:20]},
    'sentiment_analysis': {
        'by_rating': sentiment_by_rating.to_dict()
    },
    'recommendations': [
        'Focus on reviews with ratings 1-3 for improvement opportunities',
        f'Customer satisfaction rate is {positive_pct:.1f}% - monitor trends',
        'Extract and analyze common complaint phrases for product issues',
        'Leverage positive reviews for marketing and testimonials'
    ]
}

print(json.dumps(insights['dataset_summary'], indent=2))

In [ ]:
# Visualize insights
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sentiment distribution
axes[0, 0].pie([negative_count, positive_count], labels=['Negative', 'Positive'],
              colors=['#d62728', '#2ca02c'], autopct='%1.1f%%')
axes[0, 0].set_title('Overall Sentiment Distribution')

# 2. Rating distribution
axes[0, 1].bar(df['overall'].value_counts().sort_index().index,
              df['overall'].value_counts().sort_index().values,
              color='steelblue', edgecolor='black')
axes[0, 1].set_xlabel('Rating')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Rating Distribution')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Top words
top_phrases_list = [p[0] for p in top_bigrams[:10]]
top_counts_list = [p[1] for p in top_bigrams[:10]]
axes[1, 0].barh(top_phrases_list, top_counts_list, color='teal')
axes[1, 0].set_xlabel('Frequency')
axes[1, 0].set_title('Top 10 Phrases')
axes[1, 0].invert_yaxis()

# 4. Sentiment by rating
rating_sentiment = df.groupby('overall')['sentiment'].mean() * 100
axes[1, 1].bar(rating_sentiment.index, rating_sentiment.values, color=['red', 'orange', 'yellow', 'lightgreen', 'green'])
axes[1, 1].set_xlabel('Rating')
axes[1, 1].set_ylabel('% Positive Sentiment')
axes[1, 1].set_title('Sentiment by Rating')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save comprehensive report
report_path = '../assets/reports/insights_report.json'
Path(report_path).parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(insights, f, indent=2, default=str)

print(f'Insights report saved to {report_path}')
print('\n=== FINAL INSIGHTS ===')
for rec in insights['recommendations']:
    print(f'  • {rec}')

In [ ]:
# Export processed data sample
sample_output = df[['reviewText_clean', 'overall', 'sentiment', 'text_length']].head(100)
sample_path = '../assets/reports/processed_reviews_sample.csv'
sample_output.to_csv(sample_path, index=False)

print(f'Sample processed reviews exported to {sample_path}')
print(f'\nDataset Summary:')
print(f'  Total records: {len(df)}')
print(f'  Positive: {(df["sentiment"] == 1).sum()}')
print(f'  Negative: {(df["sentiment"] == 0).sum()}')
print(f'  Average rating: {df["overall"].mean():.2f}/5')